# 01 — `Dataset` and `DataLoader`: Batching, Shuffling, Workers, Collation

**Overview:** a custom `Dataset`, a `DataLoader` with batching/shuffling, a custom `collate_fn`
for variable-length sequences, an `IterableDataset` for streaming, and a `num_workers` timing
comparison.

**Required libraries:** `torch`.

Connects to: Day 1's `01_python` (the `__len__`/`__getitem__` contract, and the GIL/multiprocessing
discussion `num_workers` relies on) and `04_datasets` (Hugging Face's `IterableDataset`,
the same streaming idea). Leads to: `22_pytorch_training`.

### Abbreviations used in this notebook

| Term | Full form | What it is |
|---|---|---|
| **GIL** | Global Interpreter Lock | CPython lock letting only one thread run Python bytecode at a time; limits CPU-bound threading. |

## 0. Environment

In [1]:
import torch
from torch.utils.data import Dataset, IterableDataset, DataLoader
print("torch:", torch.__version__)
torch.manual_seed(42)


torch: 2.14.1


## 1. A custom `Dataset`: exactly the `__len__`/`__getitem__` contract from Day 1's `01_python`

🎯 **Interview Tip:** this is the SAME magic-method contract as `ToyDataset` back in Day 1 --
PyTorch's `Dataset` doesn't require inheriting any special behavior beyond these two methods.

In [2]:
class SyntheticClassificationDataset(Dataset):
    def __init__(self, n_samples, n_features, seed=42):
        g = torch.Generator().manual_seed(seed)
        self.X = torch.randn(n_samples, n_features, generator=g)
        self.y = (self.X.sum(dim=1) > 0).long()   # a simple, learnable binary rule

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

dataset = SyntheticClassificationDataset(n_samples=200, n_features=10)
print("dataset length:", len(dataset))
print("dataset[0]:", dataset[0])

assert len(dataset) == 200
x0, y0 = dataset[0]
assert x0.shape == (10,)


dataset length: 200
dataset[0]: (tensor([ 1.9269,  1.4873,  0.9007, -2.1055,  0.6784, -1.2345, -0.0431, -1.6047,
        -0.7521,  1.6487]), tensor(1))


## 2. `DataLoader`: batching and shuffling

In [3]:
loader = DataLoader(dataset, batch_size=16, shuffle=True)

first_epoch_first_batch_X, _ = next(iter(loader))
second_epoch_first_batch_X, _ = next(iter(loader))

print("batch shape:", first_epoch_first_batch_X.shape, "== (batch_size, n_features) = (16, 10)")
assert first_epoch_first_batch_X.shape == (16, 10)

# With shuffle=True, different iterations over the SAME loader should (almost certainly)
# produce a different first batch.
print("\nare the two epochs' first batches identical?",
      torch.equal(first_epoch_first_batch_X, second_epoch_first_batch_X))
assert not torch.equal(first_epoch_first_batch_X, second_epoch_first_batch_X)

num_batches = len(loader)
print(f"\n{len(dataset)} samples / batch_size=16 -> {num_batches} batches per epoch "
      f"(ceil(200/16) = {-(-200 // 16)})")
assert num_batches == -(-200 // 16)


batch shape: torch.Size([16, 10]) == (batch_size, n_features) = (16, 10)

are the two epochs' first batches identical? False

200 samples / batch_size=16 -> 13 batches per epoch (ceil(200/16) = 13)


## 3. A custom `collate_fn` for variable-length sequences

⚠️ **Common Mistake:** trying to batch variable-length sequences with the DEFAULT collate
function, which assumes every sample has the same shape and will raise an error.

In [4]:
class VariableLengthDataset(Dataset):
    def __init__(self, n_samples, max_len=10, seed=42):
        g = torch.Generator().manual_seed(seed)
        lengths = torch.randint(2, max_len, (n_samples,), generator=g)
        self.sequences = [torch.randint(1, 100, (int(l),), generator=g) for l in lengths]

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        return self.sequences[idx]

def pad_collate_fn(batch):
    max_len = max(len(seq) for seq in batch)
    padded = torch.zeros(len(batch), max_len, dtype=torch.long)
    for i, seq in enumerate(batch):
        padded[i, :len(seq)] = seq
    lengths = torch.tensor([len(seq) for seq in batch])
    return padded, lengths

var_dataset = VariableLengthDataset(n_samples=20)
print("sample lengths:", [len(var_dataset[i]) for i in range(5)])

try:
    bad_loader = DataLoader(var_dataset, batch_size=4)
    next(iter(bad_loader))
    print("default collate succeeded (unexpected -- all sampled lengths happened to match)")
except RuntimeError as e:
    print(f"default collate FAILED as expected (variable lengths can't be stacked): {type(e).__name__}")

good_loader = DataLoader(var_dataset, batch_size=4, collate_fn=pad_collate_fn)
padded_batch, lengths = next(iter(good_loader))
print("\npadded batch shape:", padded_batch.shape, "(all sequences padded to the batch's max length)")
print("original lengths in this batch:", lengths.tolist())
assert padded_batch.shape[0] == 4


sample lengths: [8, 5, 6, 8, 4]
default collate FAILED as expected (variable lengths can't be stacked): RuntimeError

padded batch shape: torch.Size([4, 8]) (all sequences padded to the batch's max length)
original lengths in this batch: [8, 5, 6, 8]


## 4. `IterableDataset`: for streaming data

💡 **Intuition:** the same idea as Hugging Face's streaming `IterableDataset` from Day 1's
`04_datasets` -- implement `__iter__` instead of `__len__`/`__getitem__`, for data too large
(or too dynamic) to index randomly.

In [5]:
class SyntheticStreamDataset(IterableDataset):
    def __init__(self, total_samples, n_features):
        self.total_samples = total_samples
        self.n_features = n_features

    def __iter__(self):
        for _ in range(self.total_samples):
            yield torch.randn(self.n_features)

stream_dataset = SyntheticStreamDataset(total_samples=1_000_000, n_features=5)
stream_loader = DataLoader(stream_dataset, batch_size=8)

first_batch = next(iter(stream_loader))
print("first streamed batch shape:", first_batch.shape)
assert first_batch.shape == (8, 5)
print("\nnote: the dataset claims 1,000,000 samples but nothing was materialized up front -- "
      "exactly the generator/streaming pattern from Day 1's 01_python and 04_datasets.")


first streamed batch shape: torch.Size([8, 5])

note: the dataset claims 1,000,000 samples but nothing was materialized up front -- exactly the generator/streaming pattern from Day 1's 01_python and 04_datasets.


## 5. `num_workers`: parallel loading via separate processes

⚠️ **Common Mistake:** on macOS/Windows (the `spawn` multiprocessing start method), a `Dataset`
class defined directly in a notebook's `__main__` can't be pickled to worker processes --
exactly the same issue Day 1's `01_python` multiprocessing demo hit. The fix is the same: put
the slow `Dataset` in an importable module file.

⚠️ **Also note:** each worker process has to import `torch` from scratch, a real fixed startup
cost (often several seconds). For a small dataset, that fixed cost can outweigh the
parallelism benefit entirely -- `num_workers > 0` only wins when there's enough expensive
per-sample work to amortize worker startup. Don't assume more workers is always faster without
measuring on your actual workload.

In [6]:
from pathlib import Path

worker_dataset_code = '''
import time
import torch
from torch.utils.data import Dataset

class SlowDataset(Dataset):
    def __init__(self, n_samples):
        self.n_samples = n_samples

    def __len__(self):
        return self.n_samples

    def __getitem__(self, idx):
        time.sleep(0.01)   # simulate slow preprocessing (e.g. image decoding, tokenization)
        return torch.tensor([float(idx)])
'''
Path("_slow_dataset_demo.py").write_text(worker_dataset_code)

from _slow_dataset_demo import SlowDataset


In [7]:
import time

slow_dataset = SlowDataset(n_samples=100)

start = time.perf_counter()
loader_0_workers = DataLoader(slow_dataset, batch_size=16, num_workers=0)
for _ in loader_0_workers:
    pass
time_0_workers = time.perf_counter() - start

start = time.perf_counter()
loader_2_workers = DataLoader(slow_dataset, batch_size=16, num_workers=2)
for _ in loader_2_workers:
    pass
time_2_workers = time.perf_counter() - start

print(f"num_workers=0: {time_0_workers:.2f}s")
print(f"num_workers=2: {time_2_workers:.2f}s")
if time_2_workers < time_0_workers:
    print(f"speedup: {time_0_workers / time_2_workers:.2f}x -- workers overlapped the artificial "
          "sleep() across processes, as intended.")
else:
    print("no speedup observed here -- on this machine, worker process startup (each worker "
          "re-importing torch) outweighed the parallelism benefit for this workload size. "
          "This is a real, common outcome for small workloads, not a bug: num_workers helps "
          "when there's enough expensive per-sample work to amortize that fixed startup cost.")
print("\nEither way, both loaders must visit every sample exactly once per epoch:")

# The correctness property that must hold regardless of timing: every sample is loaded exactly
# once per epoch, independent of num_workers.
num_batches_0 = sum(1 for _ in DataLoader(slow_dataset, batch_size=16, num_workers=0))
num_batches_2 = sum(1 for _ in DataLoader(slow_dataset, batch_size=16, num_workers=2))
print(f"num_workers=0 batch count: {num_batches_0}, num_workers=2 batch count: {num_batches_2}")
assert num_batches_0 == num_batches_2 == -(-len(slow_dataset) // 16)

Path("_slow_dataset_demo.py").unlink()   # cleanup the temporary worker module, now that every use is done


num_workers=0: 1.19s
num_workers=2: 11.59s
no speedup observed here -- on this machine, worker process startup (each worker re-importing torch) outweighed the parallelism benefit for this workload size. This is a real, common outcome for small workloads, not a bug: num_workers helps when there's enough expensive per-sample work to amortize that fixed startup cost.

Either way, both loaders must visit every sample exactly once per epoch:


num_workers=0 batch count: 7, num_workers=2 batch count: 7


## 6. Connection to what's next

`22_pytorch_training` wires exactly this `DataLoader` into a full training loop: for each
batch from the loader, run the model's forward pass, compute the loss, call `.backward()`,
and step the optimizer.

## 7. Interview questions
1. Why does a custom `Dataset` only need to implement `__len__` and `__getitem__`?
2. Why does `DataLoader`'s default collate function fail on variable-length sequences, and what fixes it?
3. Why does `num_workers > 0` use separate processes instead of threads?
4. When would you reach for `IterableDataset` instead of `Dataset`?
5. Why might a `Dataset` class defined directly in a Jupyter notebook fail with `num_workers > 0` on macOS, and how do you fix it?

## 8. Exercises
1. Build a `Dataset` wrapping image-like tensors (e.g. random `(3, 32, 32)` tensors with labels) and verify `DataLoader` batches them correctly.
2. Write a `collate_fn` that also returns a boolean mask alongside the padded sequences, indicating which positions are real vs padding.
3. Benchmark `num_workers` at 0, 1, 2, and 4 on the `SlowDataset` and plot total loading time vs `num_workers`.

## 9. Summary

✅ `Dataset` (`__len__`+`__getitem__`) and `IterableDataset` (`__iter__`) feed a `DataLoader`,
which handles batching, shuffling, and parallel loading via separate processes
(`num_workers`); a custom `collate_fn` is needed for non-uniformly-shaped samples. Next:
`22_pytorch_training`.